In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import asyncio
import pandas as pd
from labellers.label_transcript import label_transcript_with_topics

# Change only this value to process a different company.
COMPANY = "ASML"
TRANSCRIPT_DIR = Path(f"data/Transcripts/{COMPANY}")
TOPIC_CSV_DIR = Path(f"data/topic_dataset/{COMPANY}")
COOLDOWN_SECONDS = 60

transcript_files = sorted(TRANSCRIPT_DIR.glob("*.txt"))

print(f"Found {len(transcript_files)} {COMPANY} transcripts")
for idx, transcript_file in enumerate(transcript_files):
    print(f"Processing {transcript_file.name}")
    await label_transcript_with_topics(str(transcript_file))
    if idx < len(transcript_files) - 1:
        print(f"Cooldown: waiting {COOLDOWN_SECONDS} seconds before next transcript...")
        await asyncio.sleep(COOLDOWN_SECONDS)

print(f"Done processing {COMPANY} transcripts")


In [12]:
csv_files = sorted(TOPIC_CSV_DIR.glob("*.csv"))

if not csv_files:
    print(f"No CSV files found for {COMPANY} in {TOPIC_CSV_DIR}")
else:
    issues_found = False

    for csv_path in csv_files:
        print(csv_path)
        df = pd.read_csv(csv_path)

        if "topic" not in df.columns:
            issues_found = True
            print(f"\n[Missing column] {csv_path.name} has no 'topic' column.")
            continue

        # Convert to numeric; non-numeric values become NaN
        numeric_col = pd.to_numeric(df["topic"], errors="coerce")

        # Valid values are 1-10 for topic classification
        invalid_mask = numeric_col.isna() | ~numeric_col.isin(range(1, 11))

        if invalid_mask.any():
            issues_found = True
            bad_rows = df.loc[invalid_mask, ["topic"]].copy()
            bad_rows["row_index"] = bad_rows.index
            print(f"\n[Invalid values] {csv_path.name}")
            print(bad_rows[["row_index", "topic"]].to_string(index=False))

    if not issues_found:
        print(f"All {COMPANY} files passed: every 'topic' value is numeric 1-10.")

data\hedging_dataset\ASML\2016-Apr-20-ASML.csv
data\hedging_dataset\ASML\2016-Jan-20-ASML.csv
data\hedging_dataset\ASML\2016-Jul-20-ASML.csv
data\hedging_dataset\ASML\2016-Oct-19-ASML.csv
data\hedging_dataset\ASML\2017-Apr-19-ASML.csv
data\hedging_dataset\ASML\2017-Jan-18-ASML.csv
data\hedging_dataset\ASML\2017-Jul-19-ASML.csv
data\hedging_dataset\ASML\2017-Oct-18-ASML.csv
data\hedging_dataset\ASML\2018-Apr-18-ASML.csv
data\hedging_dataset\ASML\2018-Jan-17-ASML.csv
data\hedging_dataset\ASML\2018-Jul-18-ASML.csv
data\hedging_dataset\ASML\2018-Oct-17-ASML.csv
data\hedging_dataset\ASML\2019-Apr-17-ASML.csv
data\hedging_dataset\ASML\2019-Jan-23-ASML.csv
data\hedging_dataset\ASML\2019-Jul-17-ASML.csv
data\hedging_dataset\ASML\2019-Oct-16-ASML.csv
data\hedging_dataset\ASML\2020-Apr-15-ASML.csv
data\hedging_dataset\ASML\2020-Jan-22-ASML.csv
data\hedging_dataset\ASML\2020-Jul-15-ASML.csv
All files passed: every 'isHedge' value is numeric 0/1 (including 0.0/1.0).
